# Pertemuan 7: Clustering (K-Means & Hierarchical)

In [1]:
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

# ============================================================
# TUGAS 1 & 2: K-Means & Hierarchical pada Iris Dataset
# ============================================================
print("=== Tugas 1 & 2: Iris Clustering Analysis ===")

# 1. Load data & Pilih 2 features (Petal Length & Petal Width)
iris = load_iris()
X_iris = iris.data[:, 2:]  # Mengambil fitur petal saja untuk visualisasi
target_iris = iris.target

# 2. Menentukan K Optimal (Elbow + Silhouette)
inertias = []
silhouettes = []
K_range = range(2, 11)

print("\nEvaluasi K-Means pada Iris:")
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_iris)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_iris, labels))
    print(f"  K={k:2d} → Inertia: {km.inertia_:>8.2f}  Silhouette: {silhouettes[-1]:.3f}")

best_k = list(K_range)[np.argmax(silhouettes)]
print(f"\n→ K optimal berdasarkan Silhouette Score: K={best_k}")

# Simpan Plot Elbow & Silhouette
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, "bo-")
axes[0].set_title("Elbow Method (Iris)")
axes[0].set_xlabel("K")
axes[1].plot(K_range, silhouettes, "ro-")
axes[1].set_title("Silhouette Score (Iris)")
axes[1].set_xlabel("K")
plt.tight_layout()
plt.savefig("pertemuan07_iris_metrics.png")
plt.close()

# 3. Hierarchical Clustering (Linkage Comparison)
print("\nEvaluasi Hierarchical (Linkage Comparison):")
methods = ['single', 'complete', 'average', 'ward']
for m in methods:
    agg = AgglomerativeClustering(n_clusters=3, linkage=m)
    labels_h = agg.fit_predict(X_iris)
    score = silhouette_score(X_iris, labels_h)
    print(f"  Method: {m:8s} → Silhouette: {score:.3f}")

=== Tugas 1 & 2: Iris Clustering Analysis ===

Evaluasi K-Means pada Iris:
  K= 2 → Inertia:    86.39  Silhouette: 0.765
  K= 3 → Inertia:    31.37  Silhouette: 0.660
  K= 4 → Inertia:    19.47  Silhouette: 0.613
  K= 5 → Inertia:    13.92  Silhouette: 0.588
  K= 6 → Inertia:    11.04  Silhouette: 0.578
  K= 7 → Inertia:     9.24  Silhouette: 0.575
  K= 8 → Inertia:     7.67  Silhouette: 0.590
  K= 9 → Inertia:     6.46  Silhouette: 0.587
  K=10 → Inertia:     5.55  Silhouette: 0.420

→ K optimal berdasarkan Silhouette Score: K=2

Evaluasi Hierarchical (Linkage Comparison):
  Method: single   → Silhouette: 0.535
  Method: complete → Silhouette: 0.657
  Method: average  → Silhouette: 0.657
  Method: ward     → Silhouette: 0.657


In [2]:
# TUGAS 3: K-Means vs Hierarchical (Comparison)
# ============================================================
print("\n=== Tugas 3: Comparison (K-Means vs Hierarchical) ===")

# K-Means Final (K=3)
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = km_final.fit_predict(X_iris)

# Hierarchical Final (Ward, n=3)
agg_final = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg_final.fit_predict(X_iris)

print(f"K-Means Silhouette     : {silhouette_score(X_iris, labels_km):.3f}")
print(f"Hierarchical Silhouette: {silhouette_score(X_iris, labels_agg):.3f}")
print(f"K-Means DB Index       : {davies_bouldin_score(X_iris, labels_km):.3f}")
print(f"Hierarchical DB Index   : {davies_bouldin_score(X_iris, labels_agg):.3f}")

# Analisis: K-Means lebih efisien untuk data besar, sementara Hierarchical
# sangat bagus untuk melihat struktur hubungan antar data (dendrogram).


=== Tugas 3: Comparison (K-Means vs Hierarchical) ===
K-Means Silhouette     : 0.660
Hierarchical Silhouette: 0.657
K-Means DB Index       : 0.485
Hierarchical DB Index   : 0.481


In [3]:
# TUGAS 4: Customer Segmentation Project
# ============================================================
print("\n=== Tugas 4: Customer Segmentation (Sintetis) ===")

# Generate Synthetic Customer Data
np.random.seed(42)
income = np.concatenate([
    np.random.normal(25, 5, 50),   # Low Income
    np.random.normal(25, 5, 50),   # Low Income
    np.random.normal(80, 10, 50),  # High Income
    np.random.normal(80, 10, 50),  # High Income
])
spending = np.concatenate([
    np.random.normal(20, 5, 50),   # Low Spending
    np.random.normal(75, 5, 50),   # High Spending
    np.random.normal(20, 5, 50),   # Low Spending
    np.random.normal(80, 8, 50),   # High Spending
])
X_cust = np.column_stack([income, spending])

# Scaling
scaler = StandardScaler()
X_cust_scaled = scaler.fit_transform(X_cust)

# Clustering
km_cust = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_cust = km_cust.fit_predict(X_cust_scaled)

# Profiling
print("\nProfil Segmen Pelanggan:")
print(f"{'Cluster':>8}  {'Avg Income':>12}  {'Avg Spending':>13}  {'Count':>6}")
for c in range(4):
    mask = labels_cust == c
    print(f"{c:>8d}  {income[mask].mean():>12.1f}  {spending[mask].mean():>13.1f}  {mask.sum():>6d}")

# Penamaan Segmen & Interpretasi Bisnis
print("\nStrategi Bisnis per Segmen:")
for c in range(4):
    avg_inc = income[labels_cust == c].mean()
    avg_sp = spending[labels_cust == c].mean()

    if avg_inc > 50 and avg_sp > 50:
        name = "VIP/Premium"
        rec = "Berikan layanan personal dan akses eksklusif ke produk baru."
    elif avg_inc > 50 and avg_sp <= 50:
        name = "Potential Loyalist"
        rec = "Berikan promo bundling untuk meningkatkan nilai belanja."
    elif avg_inc <= 50 and avg_sp > 50:
        name = "Impulsive Spender"
        rec = "Berikan notifikasi flash sale dan produk tren terbaru."
    else:
        name = "Budget Conscious"
        rec = "Berikan diskon besar atau program poin loyalitas terjangkau."

    print(f"Cluster {c} ({name}):\n  -> {rec}")

# Visualisasi Segmentasi
plt.figure(figsize=(8, 6))
scatter = plt.scatter(income, spending, c=labels_cust, cmap='viridis', alpha=0.7)
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Customer Segments Visualization")
plt.colorbar(scatter, label='Cluster ID')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig("pertemuan07_customer_segmentation.png")
plt.close()

print("\n=== Kesimpulan Akhir ===")
print("1. K-Means berhasil mengelompokkan data dengan struktur sferis dengan baik.")
print("2. Silhouette Score menunjukkan pemisahan cluster yang cukup jelas pada data iris (K=3) dan customer.")
print("3. Penentuan K optimal dilakukan secara objektif dengan Elbow Method dan Silhouette Analysis.")
print("4. Segmentasi pelanggan memberikan dasar untuk pengambilan keputusan marketing yang tertarget.")

print("\n✅ File output telah disimpan:")
print("- pertemuan07_iris_metrics.png")
print("- pertemuan07_customer_segmentation.png")
print("\n✅ Tugas Pertemuan 07 Selesai.")


=== Tugas 4: Customer Segmentation (Sintetis) ===

Profil Segmen Pelanggan:
 Cluster    Avg Income   Avg Spending   Count
       0          25.1           74.9      50
       1          79.6           20.2      50
       2          23.9           20.8      50
       3          80.8           81.5      50

Strategi Bisnis per Segmen:
Cluster 0 (Impulsive Spender):
  -> Berikan notifikasi flash sale dan produk tren terbaru.
Cluster 1 (Potential Loyalist):
  -> Berikan promo bundling untuk meningkatkan nilai belanja.
Cluster 2 (Budget Conscious):
  -> Berikan diskon besar atau program poin loyalitas terjangkau.
Cluster 3 (VIP/Premium):
  -> Berikan layanan personal dan akses eksklusif ke produk baru.

=== Kesimpulan Akhir ===
1. K-Means berhasil mengelompokkan data dengan struktur sferis dengan baik.
2. Silhouette Score menunjukkan pemisahan cluster yang cukup jelas pada data iris (K=3) dan customer.
3. Penentuan K optimal dilakukan secara objektif dengan Elbow Method dan Silhouette Anal